<a href="https://colab.research.google.com/github/Vipanshi-gthb/HCL_proj/blob/main_analysis_engine/main_analysis_engine_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-genai

In [ ]:
!pip install -q google-auth==2.49.0

In [ ]:
from google import genai

In [ ]:
import google.auth

print(google.auth.__version__)

In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("Gemini_api_key")

client = genai.Client(api_key=api_key)

In [ ]:
print("Gemini client created successfully")
print(client.models.list())

In [ ]:
for model in client.models.list():
    print(model.name)

In [ ]:
# Cell A: Restore auth & import
from google.colab import userdata
from google import genai

In [ ]:
# Cell B: Initialize client from Colab Secrets
api_key = userdata.get("Gemini_api_key")
client = genai.Client(api_key=api_key)

In [ ]:
# --- STEP 3.3.1: TEST GEMINI CONNECTION ---

try:
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents="Explain what a dataset is in one sentence."
    )
    print("✅ Success! Gemini connection works:\n")
    print(response.text)

except Exception as e:
    print("⚠️ Encountered an error during call:\n", e)

In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Explain what a dataset is in one sentence."
)

print(response.text)

In [ ]:
operations = """
Available analysis operations:

1. sum
   Use when the user asks for the total of a numerical column.

2. average
   Use when the user asks for the average of a numerical column.

3. min
   Use when the user asks for the minimum value.

4. max
   Use when the user asks for the maximum value.

5. sales_by_category
   Use when the user asks about sales grouped by product category.

6. sales_by_region
   Use when the user asks about sales grouped by region.

7. profit_by_category
   Use when the user asks about profit grouped by category.

8. average_rating_by_category
   Use when the user asks about average rating by category.

9. orders_by_status
   Use when the user asks about order status counts.

10. monthly_sales
    Use when the user asks about sales over time/monthly sales.

11. best_selling_product
    Use when the user asks which product sold the highest quantity.

12. highest_revenue_product
    Use when the user asks which product generated the highest sales/revenue.
"""

In [ ]:
import pandas as pd

# Load the dataset into df
df = pd.read_csv("P123_sample_ecommerce_dataset.csv")

# Make sure Order_Date is converted to datetime format (from Step 1)
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

# Verify it loaded by checking the first few rows
df.head()

In [ ]:
schema = df.dtypes.astype(str).to_dict()
print(schema)

In [ ]:
# --- STEP 3.7: BASIC QUERY INTERPRETER ---

def interpret_question_basic(question: str):
    prompt = f"""
You are the query interpreter for a data analysis application.
Your job is NOT to calculate the answer, but to determine which operation to execute.

Available operations:
{operations}

Dataset columns and types:
{schema}

User question:
{question}

Return ONLY the operation name from the list above. Do not include extra text.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text.strip()

In [ ]:
# --- STEP 3.8: TEST BASIC INTERPRETER ---

test_question = "Which category generated the most sales?"
operation_result = interpret_question_basic(test_question)

print(f"User Question: '{test_question}'")
print(f"Selected Operation: '{operation_result}'")

In [ ]:
{
  "operation": "sum",
  "column": "Total_Sales_INR"
}

In [ ]:
# --- STEP 3.11: PYDANTIC SCHEMA ---
from pydantic import BaseModel
from typing import Optional

class QueryPlan(BaseModel):
    operation: str
    column: Optional[str] = None

In [ ]:
# --- STEP 3.12: STRUCTURED QUERY INTERPRETER ---
from google.genai import types

def interpret_question(question: str):
    prompt = f"""
You are the query interpreter for a data analysis application.
Your job is to convert the user's natural-language question into an analysis operation.

DO NOT calculate the answer.

Available operations:
{operations}

Dataset columns and types:
{schema}

User question:
{question}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=QueryPlan,
        )
    )

    # Convert the JSON string directly into a validated QueryPlan object
    return QueryPlan.model_validate_json(response.text)

In [ ]:
# --- STEP 3.13: TEST STRUCTURED QUERY PLANS ---

print("--- TEST 1 ---")
plan1 = interpret_question("Which category generated the most sales?")
print(plan1)

print("\n--- TEST 2 ---")
plan2 = interpret_question("What is the total sales?")
print(plan2)

print("\n--- TEST 3 ---")
plan3 = interpret_question("What is the average profit?")
print(plan3)

In [ ]:
# --- RE-LOAD STEP 2: PANDAS ANALYSIS ENGINE ---

def calculate_sum(df, column):
    return df[column].sum()

def calculate_average(df, column):
    return df[column].mean()

def calculate_min(df, column):
    return df[column].min()

def calculate_max(df, column):
    return df[column].max()

def sales_by_category(df):
    return df.groupby("Category")["Total_Sales_INR"].sum().sort_values(ascending=False)

def sales_by_region(df):
    return df.groupby("Region")["Total_Sales_INR"].sum().sort_values(ascending=False)

def profit_by_category(df):
    return df.groupby("Category")["Profit_INR"].sum().sort_values(ascending=False)

def average_rating_by_category(df):
    return df.groupby("Category")["Rating"].mean().sort_values(ascending=False)

def orders_by_status(df):
    return df["Order_Status"].value_counts()

def monthly_sales(df):
    return df.groupby(df["Order_Date"].dt.to_period("M"))["Total_Sales_INR"].sum()

def best_selling_product(df):
    return df.groupby("Product")["Quantity"].sum().sort_values(ascending=False).head(1)

def highest_revenue_product(df):
    return df.groupby("Product")["Total_Sales_INR"].sum().sort_values(ascending=False).head(1)

# Core router function for Pandas operations
def analyze(df, operation, column=None):
    if operation == "sum":
        return calculate_sum(df, column)
    elif operation == "average":
        return calculate_average(df, column)
    elif operation == "min":
        return calculate_min(df, column)
    elif operation == "max":
        return calculate_max(df, column)
    elif operation == "sales_by_category":
        return sales_by_category(df)
    elif operation == "sales_by_region":
        return sales_by_region(df)
    elif operation == "profit_by_category":
        return profit_by_category(df)
    elif operation == "average_rating_by_category":
        return average_rating_by_category(df)
    elif operation == "orders_by_status":
        return orders_by_status(df)
    elif operation == "monthly_sales":
        return monthly_sales(df)
    elif operation == "best_selling_product":
        return best_selling_product(df)
    elif operation == "highest_revenue_product":
        return highest_revenue_product(df)
    else:
        return f"Unknown operation: {operation}"

In [ ]:
# --- STEP 4.1: PIPELINE CONNECT FUNCTION ---

def execute_query(question: str):
    # 1. Route natural language to structured plan
    plan = interpret_question(question)

    print(f"🤖 User Question: '{question}'")
    print(f"⚙️  Query Plan   -> Operation: '{plan.operation}' | Column: '{plan.column}'")

    # 2. Execute Pandas analytical function
    result = analyze(df, operation=plan.operation, column=plan.column)

    return plan, result

In [ ]:
# --- STEP 4.2: NATURAL RESPONSE GENERATOR ---

def ask_p123(question: str):
    # Get plan and pandas execution output
    plan, result = execute_query(question)

    # Convert Pandas result to string format for Gemini context
    if hasattr(result, "to_string"):
        data_context = result.to_string()
    else:
        data_context = str(result)

    prompt = f"""
You are an expert data analysis assistant named P_123.

A user asked: "{question}"

The underlying data analysis engine performed the operation '{plan.operation}' and produced this exact result:
{data_context}

Provide a concise, polite, and professional natural language response summarizing this result clearly for the user.
Do not make up numbers—use ONLY the exact result provided.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    print("-" * 50)
    print("💬 Assistant Response:")
    print(response.text)
    print("=" * 50)

    return result

In [ ]:
ask_p123("Which category generated the most sales?")

In [ ]:
# --- STEP 5.1: PLOTTING ENGINE ---
import matplotlib.pyplot as plt
import seaborn as sns

# Set clean style
sns.set_theme(style="whitegrid")

def generate_chart(result, operation: str, question: str):
    """
    Renders an automatic chart based on the analytical result and operation type.
    """
    plt.figure(figsize=(8, 4.5))

    # 1. Categorical / Grouped Bar Charts
    if operation in ["sales_by_category", "sales_by_region", "profit_by_category"]:
        ax = sns.barplot(x=result.index, y=result.values, palette="viridis")
        plt.title(f"Visual Breakdown: {operation.replace('_', ' ').title()}", fontsize=12, fontweight='bold')
        plt.xlabel("Category / Region", fontsize=10)
        plt.ylabel("Value (INR)", fontsize=10)
        plt.xticks(rotation=15)
        plt.tight_layout()
        plt.show()

    # 2. Time Series / Line Charts
    elif operation == "monthly_sales":
        # Convert period index to string for clean plotting
        x_labels = [str(period) for period in result.index]
        plt.plot(x_labels, result.values, marker='o', color='#2b5c8f', linewidth=2.5)
        plt.title("Monthly Sales Trend", fontsize=12, fontweight='bold')
        plt.xlabel("Month", fontsize=10)
        plt.ylabel("Total Sales (INR)", fontsize=10)
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

    # 3. Distribution / Pie Charts
    elif operation == "orders_by_status":
        plt.pie(result.values, labels=result.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("pastel"))
        plt.title("Order Status Distribution", fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()

    else:
        # Scalar operations (sum, average, min, max) don't require plots
        pass

In [ ]:
# --- STEP 5.2: UPGRADED FULL PIPELINE WITH CHARTS ---

def ask_p123(question: str):
    # 1. Execute query and get plan + result
    plan, result = execute_query(question)

    # 2. Format result as string for Gemini
    if hasattr(result, "to_string"):
        data_context = result.to_string()
    else:
        data_context = str(result)

    prompt = f"""
You are an expert data analysis assistant named P_123.

A user asked: "{question}"

The underlying data analysis engine performed the operation '{plan.operation}' and produced this exact result:
{data_context}

Provide a concise, polite, and professional natural language response summarizing this result clearly for the user.
Do not make up numbers—use ONLY the exact result provided.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    # 3. Print Textual Answer
    print("-" * 50)
    print("💬 Assistant Response:")
    print(response.text)
    print("-" * 50)

    # 4. Render Chart if applicable
    generate_chart(result, plan.operation, question)
    print("=" * 50)

    return result

In [ ]:
# Test Visual 1: Grouped Bar Chart
ask_p123("Show me sales by category")

In [ ]:
# Test Visual 2: Time Series Trend
ask_p123("Show me monthly sales performance")

In [ ]:
from pydantic import BaseModel
from google.genai import types

# Define the structure Gemini must return
class CodePlan(BaseModel):
    explanation: str  # Brief summary of what the code does
    pandas_code: str  # Executable single-line or multi-line Pandas snippet

def generate_pandas_code(question: str, schema_dict: dict):
    prompt = f"""
You are a expert Python/Pandas data analyst code generator.

User Question: "{question}"

Dataset Metadata (Schema and Data Types):
{schema_dict}

Rules:
1. Assume the Pandas DataFrame is already loaded as `df`.
2. Generate valid Python Pandas code that computes the exact answer to the user's question.
3. The generated code MUST store the final output in a variable named `result`.
4. Do NOT include markdown formatting (no ```python).
5. Do NOT modify or overwrite `df`.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=CodePlan,
        )
    )

    return CodePlan.model_validate_json(response.text)

In [ ]:
def run_generated_code(pandas_code_str: str, dataframe):
    """
    Executes the generated Pandas string safely in a controlled local scope.
    """
    # Environment scope for exec()
    local_scope = {"df": dataframe, "pd": pd}

    try:
        exec(pandas_code_str, {}, local_scope)
        # Extract the resulting 'result' variable created by the generated code
        result = local_scope.get("result", "Error: 'result' variable was not set by generated code.")
        return result
    except Exception as e:
        return f"Execution Error: {str(e)}"

In [ ]:
# Extract CSV schema only (No actual row data sent to LLM)
dataset_schema = df.dtypes.astype(str).to_dict()

# Test Case
question = "Which top 3 products generated the highest total profit?"

# Step A: Get generated code from Gemini
plan = generate_pandas_code(question, dataset_schema)

print("🤖 Generated Explanation:", plan.explanation)
print("💻 Generated Pandas Code:\n", plan.pandas_code)
print("-" * 50)

# Step B: Run generated code locally on df
computed_result = run_generated_code(plan.pandas_code, df)

print("📊 Computed Local Output:")
print(computed_result)